# Past Messages — Exploratory Data Analysis

Exploratory analysis of Facebook Messenger and Instagram DM history stored in `messages.db`.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
import warnings

DB = "messages.db"
conn = sqlite3.connect(DB)

try:
    from contacts_data import YOUR_ALIASES
    YOUR_NAMES = [name for name, _ in YOUR_ALIASES]
except ImportError:
    YOUR_NAMES = ["Your Name"]
    print("WARNING: contacts_data.py not found — set YOUR_NAMES manually")

# SQL-safe IN clause, e.g. "'Tim Chen', '(◐‿◑)﻿'"
YOU = ", ".join(f"'{n}'" for n in YOUR_NAMES)

print(f"Tracking as: {YOUR_NAMES}")

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

cjk_candidates = ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK", "SimHei", "Arial Unicode MS"]
cjk_fonts = [f.name for f in fm.fontManager.ttflist if any(x in f.name for x in cjk_candidates)]
if cjk_fonts:
    plt.rcParams["font.sans-serif"] = [cjk_fonts[0], "DejaVu Sans", "Segoe UI Emoji"]
    plt.rcParams["font.family"] = "sans-serif"

warnings.filterwarnings("ignore", message="Glyph.*missing from font")
print("DB connected.")

## 1. Volume Over Time

In [ ]:
# Messages sent vs received per month
df = pd.read_sql(f"""
    SELECT
        strftime('%Y-%m', timestamp_ms/1000, 'unixepoch') AS month,
        CASE WHEN sender IN ({YOU}) THEN 'sent' ELSE 'received' END AS direction,
        COUNT(*) AS n
    FROM messages
    GROUP BY month, direction
""", conn)

pivot = df.pivot(index="month", columns="direction", values="n").fillna(0)
pivot.plot(kind="bar", stacked=True, width=0.9, color=["#4a90d9", "#e8a838"])
plt.title("Messages Sent vs Received per Month")
plt.xlabel("")
plt.xticks(range(0, len(pivot), 6), pivot.index[::6], rotation=45, ha="right")
plt.ylabel("Messages")
plt.tight_layout()
plt.show()

In [ ]:
# Activity heatmap: hour of day vs day of week
df_heat = pd.read_sql(f"""
    SELECT
        CAST(strftime('%H', timestamp_ms/1000, 'unixepoch') AS INTEGER) AS hour,
        CAST(strftime('%w', timestamp_ms/1000, 'unixepoch') AS INTEGER) AS dow,
        COUNT(*) AS n
    FROM messages
    WHERE sender IN ({YOU})
    GROUP BY hour, dow
""", conn)

heat = df_heat.pivot(index="hour", columns="dow", values="n").fillna(0)
heat.columns = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heat, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(7)); ax.set_xticklabels(heat.columns)
ax.set_yticks(range(0, 24, 2)); ax.set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 2)])
ax.set_title("When You Message (Hour x Day of Week)")
plt.colorbar(im, ax=ax, label="messages sent")
plt.tight_layout()
plt.show()

## 2. Top Contacts

In [ ]:
# Top 20 DM contacts by total messages (both directions)
df_top = pd.read_sql(f"""
    SELECT t.thread_name,
           COUNT(*) AS total,
           SUM(CASE WHEN m.sender IN ({YOU}) THEN 1 ELSE 0 END) AS sent
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE t.is_group = 0
    GROUP BY t.thread_name
    ORDER BY total DESC
    LIMIT 20
""", conn)

df_top["received"] = df_top["total"] - df_top["sent"]
df_top = df_top.sort_values("total")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_top["thread_name"], df_top["sent"],     label="sent",     color="#4a90d9")
ax.barh(df_top["thread_name"], df_top["received"], left=df_top["sent"], label="received", color="#e8a838")
ax.set_title("Top 20 DM Contacts (all time)")
ax.set_xlabel("Messages")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# New contacts per year (first message date)
df_new = pd.read_sql("""
    SELECT strftime('%Y', first_message_ts/1000, 'unixepoch') AS year, COUNT(*) AS new_contacts
    FROM threads
    WHERE is_group = 0
    GROUP BY year ORDER BY year
""", conn)

df_new.plot(x="year", y="new_contacts", kind="bar", legend=False, color="#4a90d9")
plt.title("New Contacts per Year")
plt.xlabel(""); plt.ylabel("New threads started")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Conversation Behavior

In [ ]:
# DM initiation rate per year (who breaks the silence?)
df_msgs = pd.read_sql(f"""
    SELECT m.thread_id, t.thread_name, m.sender, m.timestamp_ms,
           m.sender IN ({YOU}) AS is_you
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE m.content_type = 'text'
      AND t.is_group = 0
    ORDER BY m.thread_id, m.timestamp_ms
""", conn)

SESSION_GAP_MS = 4 * 60 * 60 * 1000

df_msgs["prev_ts"]  = df_msgs.groupby("thread_id")["timestamp_ms"].shift(1)
df_msgs["gap"]      = df_msgs["timestamp_ms"] - df_msgs["prev_ts"]
df_msgs["is_start"] = df_msgs["gap"].isna() | (df_msgs["gap"] > SESSION_GAP_MS)
df_msgs["year"]     = pd.to_datetime(df_msgs["timestamp_ms"], unit="ms").dt.year

sessions = df_msgs[df_msgs["is_start"]].copy()

# --- yearly aggregate ---
by_year = sessions.groupby("year")["is_you"].agg(["sum","count"]).reset_index()
by_year.columns = ["year","you_initiated","total"]
by_year["initiation_rate"] = by_year["you_initiated"] / by_year["total"] * 100

by_year.plot(x="year", y="initiation_rate", kind="line", marker="o", legend=False, color="#4a90d9")
plt.axhline(50, color="gray", linestyle="--", alpha=0.5)
plt.title("DM Initiation Rate per Year (% sessions you started)")
plt.ylabel("%"); plt.xlabel(""); plt.ylim(0, 100)
plt.tight_layout(); plt.show()

# --- per-contact heatmap (top 20 by session count) ---
by_contact_year = (
    sessions.groupby(["thread_name", "year"])["is_you"]
    .agg(["sum", "count"])
    .reset_index()
)
by_contact_year["rate"] = by_contact_year["sum"] / by_contact_year["count"] * 100

top_contacts = (
    by_contact_year.groupby("thread_name")["count"].sum()
    .nlargest(20).index
)
pivot_init = (
    by_contact_year[by_contact_year["thread_name"].isin(top_contacts)]
    .pivot(index="thread_name", columns="year", values="rate")
)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot_init.values, aspect="auto", cmap="RdYlBu", vmin=0, vmax=100)
ax.set_xticks(range(len(pivot_init.columns)))
ax.set_xticklabels(pivot_init.columns.astype(int), rotation=45)
ax.set_yticks(range(len(pivot_init.index)))
ax.set_yticklabels(pivot_init.index, fontsize=8)
plt.colorbar(im, ax=ax, label="% sessions you initiated")
ax.set_title("Who Initiates? Per-Contact Initiation Rate by Year\n(red=they chase you, blue=you chase them)")
plt.tight_layout(); plt.show()

In [ ]:
# Average message length (words) per year — your messages only
df_len = pd.read_sql(f"""
    SELECT strftime('%Y', timestamp_ms/1000, 'unixepoch') AS year,
           AVG(word_count) AS avg_words,
           AVG(char_count) AS avg_chars
    FROM messages
    WHERE sender IN ({YOU})
      AND content_type = 'text'
      AND word_count > 0
    GROUP BY year ORDER BY year
""", conn)

fig, ax1 = plt.subplots()
ax1.bar(df_len["year"], df_len["avg_words"], color="#4a90d9", alpha=0.7, label="avg words")
ax1.set_ylabel("Avg words per message")
ax1.set_xlabel("")
ax2 = ax1.twinx()
ax2.plot(df_len["year"], df_len["avg_chars"], color="#e8a838", marker="o", label="avg chars")
ax2.set_ylabel("Avg chars per message")
plt.title("Your Average Message Length Over Time")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.show()

## 4. Contact Drift (Who You Drifted From)

In [ ]:
# Contact Drift — Slope chart: peak year activity vs last 2 years
import datetime
import numpy as np

cutoff_ms = int((datetime.datetime.now() - datetime.timedelta(days=730)).timestamp() * 1000)

df_yearly = pd.read_sql(f"""
    SELECT t.thread_name,
           strftime('%Y', m.timestamp_ms/1000, 'unixepoch') AS year,
           COUNT(*) AS n
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE m.sender IN ({YOU})
      AND t.is_group = 0
    GROUP BY t.thread_name, year
""", conn)

peak = df_yearly.groupby("thread_name")["n"].max().rename("peak")

df_recent = pd.read_sql(f"""
    SELECT t.thread_name, COUNT(*) AS recent
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE m.sender IN ({YOU})
      AND t.is_group = 0
      AND m.timestamp_ms >= {cutoff_ms}
    GROUP BY t.thread_name
""", conn).set_index("thread_name")["recent"]

slope_df = pd.concat([peak, df_recent], axis=1).fillna(0)
slope_df = slope_df[slope_df["peak"] >= 80].copy()
slope_df = slope_df.sort_values("peak", ascending=False).head(30)

fig, ax = plt.subplots(figsize=(9, 10))
ax.set_xlim(-0.3, 1.3)
ax.set_yscale("log")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Peak Year", "Last 2 Yrs"], fontsize=12)
ax.set_yticks([])
ax.yaxis.set_minor_locator(ticker.NullLocator())

for name, row in slope_df.iterrows():
    left, right = row["peak"], max(row["recent"], 1)
    ratio = right / left if left > 0 else 1
    color = "#d94a4a" if ratio < 0.4 else "#f0a500" if ratio < 0.75 else "#4a90d9"
    ax.plot([0, 1], [left, right], color=color, linewidth=1.5, alpha=0.8)
    ax.text(-0.05, left, name, ha="right", va="center", fontsize=7.5, color=color)
    ax.text(1.05, right, f"{int(row['recent'])}", ha="left", va="center", fontsize=7.5, color=color)

ax.set_title("Contact Drift: Peak Activity vs Last 2 Years\n"
             "[red] faded >60%   [yellow] quieter   [blue] still active", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Media & Content Type Distribution

In [ ]:
# Media vs text ratio over time — your messages
df_media = pd.read_sql(f"""
    SELECT strftime('%Y', timestamp_ms/1000, 'unixepoch') AS year,
           content_type, COUNT(*) AS n
    FROM messages
    WHERE sender IN ({YOU})
    GROUP BY year, content_type
    ORDER BY year
""", conn)

pivot_media = df_media.pivot(index="year", columns="content_type", values="n").fillna(0)
pivot_media_pct = pivot_media.div(pivot_media.sum(axis=1), axis=0) * 100

pivot_media_pct.plot(kind="bar", stacked=True, width=0.85,
                     colormap="tab10", figsize=(14, 5))
plt.title("Your Message Type Mix per Year (%)")
plt.xlabel(""); plt.ylabel("%")
plt.xticks(rotation=0)
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Group vs DM Activity Over Time

In [ ]:
# Your messages in groups vs DMs per year
df_gd = pd.read_sql(f"""
    SELECT strftime('%Y', m.timestamp_ms/1000, 'unixepoch') AS year,
           t.is_group,
           COUNT(*) AS n
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE m.sender IN ({YOU})
    GROUP BY year, t.is_group
    ORDER BY year
""", conn)

df_gd["type"] = df_gd["is_group"].map({0: "DM", 1: "Group"})
pivot_gd = df_gd.pivot(index="year", columns="type", values="n").fillna(0)
pivot_gd_pct = pivot_gd.div(pivot_gd.sum(axis=1), axis=0) * 100

pivot_gd_pct.plot(kind="bar", stacked=True, color=["#4a90d9","#9b59b6"], width=0.8)
plt.title("Your Messages: Group vs DM per Year (%)")
plt.xlabel(""); plt.ylabel("%")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Response Latency

In [ ]:
# Response latency: how long does it take you vs others to reply?
import numpy as np

df_lat = pd.read_sql(f"""
    SELECT thread_id, sender, timestamp_ms,
           sender IN ({YOU}) AS is_you
    FROM messages
    WHERE content_type = 'text'
    ORDER BY thread_id, timestamp_ms
""", conn)

df_lat["prev_sender"] = df_lat.groupby("thread_id")["sender"].shift(1)
df_lat["prev_ts"]     = df_lat.groupby("thread_id")["timestamp_ms"].shift(1)
df_lat["gap_s"]       = (df_lat["timestamp_ms"] - df_lat["prev_ts"]) / 1000

# A reply = sender changed and gap < 6 hours
replies = df_lat[
    (df_lat["sender"] != df_lat["prev_sender"]) &
    (df_lat["gap_s"] > 0) &
    (df_lat["gap_s"] < 6 * 3600)
].copy()

your_replies  = replies[replies["is_you"] == 1]["gap_s"]
their_replies = replies[replies["is_you"] == 0]["gap_s"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, data, label, color in zip(
    axes,
    [your_replies, their_replies],
    ["Your reply latency", "Their reply latency"],
    ["#4a90d9", "#e8a838"]
):
    log_data = np.log10(data.clip(lower=1))
    ax.hist(log_data, bins=60, color=color, alpha=0.8, edgecolor="none")
    ax.set_xlabel("log10(seconds)")
    ax.set_ylabel("count")
    ax.set_title(label)
    # reference lines
    for secs, lbl in [(60,"1m"), (600,"10m"), (3600,"1h"), (21600,"6h")]:
        ax.axvline(np.log10(secs), color="gray", linestyle="--", alpha=0.4)
        ax.text(np.log10(secs)+0.03, ax.get_ylim()[1]*0.95, lbl, fontsize=7, color="gray")

    med = data.median()
    print(f"{label}: median {med/60:.1f} min, mean {data.mean()/60:.1f} min")

plt.suptitle("Reply Latency Distribution (log scale, replies < 6h)")
plt.tight_layout()
plt.show()

## 8. Emoji & Reactions

In [ ]:
# Emoji frequency in your messages — displayed as table (matplotlib can't render color emoji)
import re
from collections import Counter

df_emoji_src = pd.read_sql(f"""
    SELECT content FROM messages
    WHERE sender IN ({YOU})
      AND content IS NOT NULL
      AND content_type = 'text'
""", conn)

EMOJI_RE = re.compile(
    "[\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F000-\U0001F02F"
    "\U0001F0A0-\U0001F0FF"
    "\U0001F900-\U0001F9FF"
    "\U00002702-\U000027B0]+",
    flags=re.UNICODE
)

flat = [ch for text in df_emoji_src["content"].dropna()
           for match in EMOJI_RE.findall(text)
           for ch in match]

top_emoji = Counter(flat).most_common(30)
df_emoji_table = pd.DataFrame(top_emoji, columns=["emoji", "count"])
df_emoji_table.index += 1
df_emoji_table

In [ ]:
# Reaction asymmetry: who gives vs receives reactions (top 20 contacts)
df_rx = pd.read_sql(f"""
    SELECT
        r.reactor,
        m.sender,
        r.emoji,
        m.sender IN ({YOU}) AS msg_is_yours,
        r.reactor IN ({YOU}) AS reactor_is_you
    FROM reactions r
    JOIN messages m ON r.message_id = m.id
""", conn)

# reactions YOU gave to others vs reactions others gave to YOUR messages
you_gave     = df_rx[df_rx["reactor_is_you"] == 1].groupby("sender").size().rename("you_gave")
you_received = df_rx[(df_rx["msg_is_yours"] == 1) & (df_rx["reactor_is_you"] == 0)].groupby("reactor").size().rename("you_received")

rx_summary = pd.concat([you_gave, you_received], axis=1).fillna(0)
rx_summary = rx_summary[(rx_summary["you_gave"] + rx_summary["you_received"]) > 5]
rx_summary = rx_summary.sort_values("you_received", ascending=False).head(20)
rx_summary = rx_summary.sort_values("you_received")

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(rx_summary.index, rx_summary["you_gave"],     label="You reacted to them", color="#4a90d9")
ax.barh(rx_summary.index, rx_summary["you_received"], left=rx_summary["you_gave"], label="They reacted to you", color="#e8a838")
ax.set_title("Reaction Exchange (top contacts)")
ax.set_xlabel("Reactions")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Call Analytics

In [ ]:
# Call volume and duration over time
df_calls = pd.read_sql("""
    SELECT strftime('%Y', timestamp_ms/1000, 'unixepoch') AS year,
           COUNT(*) AS call_count,
           SUM(CASE WHEN call_duration IS NOT NULL THEN 1 ELSE 0 END) AS answered,
           AVG(CASE WHEN call_duration > 0 THEN call_duration ELSE NULL END) AS avg_duration_s,
           SUM(CASE WHEN call_duration > 0 THEN call_duration ELSE 0 END) / 3600.0 AS total_hours
    FROM messages
    WHERE content_type = 'call'
    GROUP BY year ORDER BY year
""", conn)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(df_calls["year"], df_calls["call_count"], color="#4a90d9")
axes[0].set_title("Calls per Year")
axes[0].set_ylabel("Count")

axes[1].bar(df_calls["year"], df_calls["avg_duration_s"] / 60, color="#e8a838")
axes[1].set_title("Avg Call Duration (min)")
axes[1].set_ylabel("Minutes")

axes[2].bar(df_calls["year"], df_calls["total_hours"], color="#9b59b6")
axes[2].set_title("Total Call Hours per Year")
axes[2].set_ylabel("Hours")

for ax in axes:
    ax.set_xlabel("")

plt.suptitle("Call Analytics Over Time")
plt.tight_layout()
plt.show()

# Top contacts by call time
df_call_contacts = pd.read_sql("""
    SELECT t.thread_name,
           COUNT(*) AS calls,
           SUM(CASE WHEN call_duration > 0 THEN call_duration ELSE 0 END) / 60.0 AS total_minutes
    FROM messages m
    JOIN threads t ON m.thread_id = t.thread_id
    WHERE m.content_type = 'call'
      AND t.is_group = 0
    GROUP BY t.thread_name
    ORDER BY total_minutes DESC
    LIMIT 15
""", conn).sort_values("total_minutes")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_call_contacts["thread_name"], df_call_contacts["total_minutes"], color="#9b59b6")
ax.set_title("Top 15 Contacts by Total Call Time")
ax.set_xlabel("Minutes")
plt.tight_layout()
plt.show()